# 05 · Reproduced results vs. paper
Reads every `summary.json` produced by notebooks 02-03 and puts them next to the numbers reported in the paper (Tables 5-6 / abstract).

In [ ]:
# ---- Colab setup (run first; if you ever see Keras-3 errors: Runtime > Restart session, then re-run) ----
import os, sys
os.environ["TF_USE_LEGACY_KERAS"] = "1"          # the paper's code style needs Keras 2
!pip -q install tf_keras
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/firenet"       # everything lives here, so a disconnect loses nothing
sys.path.insert(0, PROJECT)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from types import SimpleNamespace
from IPython.display import Image, display
import firenet as fn                             # written to Drive by 00_setup_and_data.ipynb
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
PAPER = {  # (dataset, model): accuracy, precision, recall, f1, jaccard, mcc   [%]  (Table 6 + Sec. IV-C)
 ("A","firenet"):(97.96,98.22,98.41,98.31,96.68,96.45), ("B","firenet"):(99.05,99.41,98.28,98.84,97.71,98.19),
 ("A","vgg16"):(95.30,95.51,95.79,95.65,91.67,90.89),   ("B","vgg16"):(97.51,96.92,97.04,96.98,94.14,95.19),
 ("A","vgg19"):(95.24,95.80,95.64,95.72,91.81,91.04),   ("B","vgg19"):(97.93,97.28,97.57,97.42,94.98,95.88),
 ("A","inceptionv3"):(97.53,98.13,97.58,97.86,95.80,95.52), ("B","inceptionv3"):(98.88,98.96,98.54,98.75,97.53,98.01)}
cols = ["accuracy", "precision", "recall", "f1", "jaccard", "mcc"]
rows = []
for (d, model), v in PAPER.items():
    rows.append(dict(dataset=d, model=model, source="paper", **dict(zip(cols, v))))
    for mode in ("paper", "strict"):
        p = f"{PROJECT}/runs/{d}_{mode}_{model}/summary.json"
        if os.path.exists(p):
            sj = pd.read_json(p, typ="series")
            rows.append(dict(dataset=d, model=model, source=f"ours ({mode} split)", **{c: round(100 * sj[c], 2) for c in cols}))
df = pd.DataFrame(rows).sort_values(["dataset", "model", "source"], ascending=[True, True, False]).reset_index(drop=True)
df.to_csv(f"{PROJECT}/runs/results_vs_paper.csv", index=False); df

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for a, d in zip(ax, ("A", "B")):
    sub = df[df.dataset == d].pivot(index="model", columns="source", values="accuracy")
    sub.plot.bar(ax=a, rot=0); a.set_title(f"Dataset {d} - validation accuracy (%)"); a.set_ylim(80, 100); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

### How to read this
* **ours (paper split)** should land close to the paper if the implementation is faithful.
* **ours (strict split)** is the leakage-free estimate: synthetic images never appear in validation. A clear gap between the two rows means the paper's number is partly driven by near-duplicate / synthetic images in the validation folds - especially on Dataset B (frames from only 25 videos).
* Remaining differences can come from details the paper does not state (exact baseline heads, image preprocessing, which epoch's weights are scored, random seeds).